# Smoke-эксперимент: маршрутизация банковских обращений

## tl;dr

На 10 строках synthetic test фактический прогон ниже даёт macro-F1 = 1.000 и top-3 accuracy = 1.000. Validation-порог abstention равен 0.562; на test coverage = 0.100 и selective accuracy = 1.000. Эти числа нельзя переносить на реальные обращения.

## Context & Methods

Notebook импортирует реальный код из `src/`, читает committed synthetic CSV и вызывает `train()` и `evaluate()`. Внутри каждого calibration fold независимо обучается word/char TF-IDF pipeline, после чего validation определяет порог передачи обращения человеку.

### Key Assumptions

- seed равен 42, split стратифицирован по intent;
- sigmoid-калибровка выполняется только внутри train;
- минимальное validation coverage задано равным 0.60;
- synthetic-примеры не являются строками BANKING77, сеть не используется.

In [1]:
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd

SEED = 42
MINIMUM_VALIDATION_COVERAGE = 0.60
PROJECT_ROOT = next(
    path for path in (Path.cwd(), Path.cwd().parent)
    if (path / 'pyproject.toml').exists()
)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from banking_ticket_routing.data import load_csv, stratified_split  # noqa: E402
from banking_ticket_routing.evaluate import evaluate  # noqa: E402
from banking_ticket_routing.service import RoutingService  # noqa: E402
from banking_ticket_routing.train import train as train_model  # noqa: E402

DATA_PATH = PROJECT_ROOT / 'data' / 'smoke.csv'

## Data

Проверяем схему, отсутствие пересечения ID и представленность всех пяти synthetic intent-классов.

In [2]:
tickets = load_csv(DATA_PATH)
train_frame, validation_frame, test_frame, split_manifest = stratified_split(
    tickets, random_state=SEED
)
assert set(train_frame['ticket_id']).isdisjoint(validation_frame['ticket_id'])
assert set(train_frame['ticket_id']).isdisjoint(test_frame['ticket_id'])
assert set(validation_frame['ticket_id']).isdisjoint(test_frame['ticket_id'])

pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'rows': [len(train_frame), len(validation_frame), len(test_frame)],
    'intents': [part['intent'].nunique() for part in (train_frame, validation_frame, test_frame)],
})

,split,rows,intents
0,train,30,5
1,validation,10,5
2,test,10,5


## Results

Обучаем и калибруем реальный pipeline, сохраняем bundle и оцениваем test, который не участвовал в выборе порога.

In [3]:
temporary_directory = TemporaryDirectory(prefix='banking-routing-smoke-')
artifact_dir = Path(temporary_directory.name) / 'artifacts'
training_metadata = train_model(
    DATA_PATH,
    artifact_dir,
    random_state=SEED,
    minimum_coverage=MINIMUM_VALIDATION_COVERAGE,
)
test_metrics = evaluate(artifact_dir / 'model.joblib', artifact_dir / 'test.csv')

pd.Series({
    'test_rows': test_metrics['rows'],
    'macro_f1': test_metrics['macro_f1'],
    'top_3_accuracy': test_metrics['top_3_accuracy'],
    'log_loss': test_metrics['log_loss'],
    'abstention_threshold': test_metrics['abstention_threshold'],
    'test_coverage': test_metrics['coverage'],
    'test_abstention_rate': test_metrics['abstention_rate'],
    'selective_accuracy': test_metrics['selective_accuracy'],
}, name='synthetic smoke')

test_rows               10.000000
macro_f1                 1.000000
top_3_accuracy           1.000000
log_loss                 0.825667
abstention_threshold     0.561539
test_coverage            0.100000
test_abstention_rate     0.900000
selective_accuracy       1.000000
Name: synthetic smoke, dtype: float64

In [4]:
routing_service = RoutingService.from_path(artifact_dir / 'model.joblib')
sample_route = routing_service.route('Where is my new card?', top_k=3)
sample_route

{'intent': 'card_arrival',
 'suggested_intent': 'card_arrival',
 'confidence': 0.6449216922449122,
 'abstained': False,
 'abstention_threshold': 0.5615392248223485,
 'candidates': [{'intent': 'card_arrival', 'probability': 0.6449216922449122},
  {'intent': 'cash_withdrawal', 'probability': 0.15910496167066926},
  {'intent': 'card_payment_wrong_exchange_rate',
   'probability': 0.08303224325981673}]}

## Takeaways

- На простом synthetic test argmax-классификация достигает macro-F1 и top-3 accuracy 1.000.
- Test coverage = 0.100 заметно ниже ограничения 0.60, заданного для validation; это ожидаемый сигнал необходимости отдельного мониторинга abstention после переноса.
- Калиброванная уверенность не является гарантией качества, а abstention означает ручную маршрутизацию, не отказ клиенту.